# 01 — Download Data from MAST

**Purpose:** Query MAST for all WFC3/IR FLT exposures from HST proposal 14706 (PI: Glikman),
confirm the expected quasar targets are returned, and download the files into
`data/raw/<target_name>/` using real target names from MAST.

**Before running:** Read `HANDOFF.md` and `CLAUDE.md` for current project state.

**Outputs:** FLT files in `data/raw/<target_name>/` (one folder per quasar, named from MAST)

**Next step:** Run `/validate-downloads`, then open `02_inspect_raw.ipynb`.

---
**IMPORTANT — Query parameters:**
- `project='HST'` (not `'HAP'`) — HAP returns processed mosaics, not individual exposures
- `productSubGroupDescription='FLT'` — calibrated individual exposures from CALWF3
- `proposal_id='14706'`
- `instrument_name='WFC3/IR'`

## Imports

In [ ]:
import os
import shutil
import glob
from pathlib import Path
from astropy.io import fits
from astroquery.mast import Observations

## Define paths

In [ ]:
RAW_DIR = Path('../data/raw')

## Query MAST

In [ ]:
obs_table = Observations.query_criteria(
    proposal_id='14706',
    instrument_name='WFC3/IR',
    project='HST'
)

print(f'Observations found: {len(obs_table)}')
print(obs_table['target_name', 'obs_id', 'filters', 't_exptime'])

In [ ]:
# Remove HAP sv/mv mosaic rows — keep only standard HST pipeline observations
mask = ['hst_' not in obs_id.lower() for obs_id in obs_table['obs_id']]
obs_table = obs_table[mask]
print(f'Observations after removing HAP sv/mv products: {len(obs_table)}')

## Confirm 6 distinct targets

In [ ]:
targets = sorted(set(obs_table['target_name']))
n = len(targets)

print(f'Distinct targets found: {n}')
for t in targets:
    print(f'  {t}')

if n != 6:
    print(f'\nWARNING: Expected 6 targets, got {n}. Review the list above before proceeding.')
else:
    print('\nAll 6 targets confirmed.')

## Get product list and filter to FLT files

In [ ]:
products = Observations.get_product_list(obs_table)

filtered = Observations.filter_products(
    products,
    productSubGroupDescription='FLT',
    project='CALWF3'
)

# Join target_name from obs_table onto filtered products
obs_id_to_target = {row['obs_id']: row['target_name'] for row in obs_table}
filtered['target_name'] = [obs_id_to_target.get(oid, 'UNKNOWN') for oid in filtered['obs_id']]

print(f'FLT files to download: {len(filtered)}')
filtered['target_name', 'productFilename', 'size']

In [ ]:
Observations.download_products(filtered, cache=True)

## Move files into per-target directories

In [ ]:
# Read TARGNAME directly from each FITS header and move into per-target directories
for flt in glob.glob('./mastDownload/HST/*/*flt.fits'):
    flt_path = Path(flt)
    target = fits.getval(str(flt_path), 'TARGNAME', ext=0)
    dest_dir = RAW_DIR / target
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / flt_path.name
    shutil.move(str(flt_path), str(dest))
    print(f"Moved {flt_path.name} → {dest}")

# shutil.rmtree('mastDownload/')  # uncomment after verifying all files moved correctly

## Next steps

1. Run `/validate-downloads` to confirm all files are present and uncorrupted
2. Open `02_inspect_raw.ipynb`